In [1]:
import sys
sys.path.append("C:\\Users\\NikonTE300CE\\Desktop\\automated-sca\\src")

from stage import Stage
from plate import Plate
from chip import Chip
import serial

import calib

In [3]:
c = Chip()
p = Plate()
# p = Plate(2,3,40000) # roughly the diameter for 6-well plate, a little off
ser = serial.Serial(port="COM4", baudrate=9600, timeout=0.1)
s = Stage(p, c, ser)

In [15]:
# First move using joystick to origin
# Then click this
s.calibOrigin()

In [9]:
s.moveToOrigin()

In [16]:
# Move stage to see first channel on camera
a = s.getStageXY()
print("First channel stage position:", a)

firstchannelpos = tuple(a)
secondchannelpos = (firstchannelpos[0], firstchannelpos[1]  + 260)

# save to tuple to do calculations later
print("First channel position set to:", firstchannelpos)
print(secondchannelpos)

First channel stage position: [-77418, -34524]
First channel position set to: (-77418, -34524)
(-77418, -34264)


In [17]:
# Move stage to see first well on camera
b = s.getStageXY()
print("First well stage position:", b)

firstwellpos = tuple(b)
# save to tuple to do calculations later
print("First well position set to:", firstwellpos)


First well stage position: [-133645, -8905]
First well position set to: (-133645, -8905)


In [ ]:
import time
from plate import Plate, Plate96

diam = 9000  # distance between wells in microns for 96-well plate 
plate = Plate96()  # or Plate6(), Plate12(), etc.
origin_xy = firstwellpos  # Example: set this to your calibrated A1 position

def well_id_to_index(well_id):
    row = ord(well_id[0].upper()) - ord('A')
    col = int(well_id[1:]) - 1
    return row, col

def get_well_xy(plate, well_id, origin_xy=(0, 0)):
    row, col = well_id_to_index(well_id)
    x = origin_xy[0] + col * diam
    y = origin_xy[1] + row * diam
    return (x, y)   


well_ids = ["A1", "B1"]
channel_positions = [firstchannelpos, secondchannelpos]  # make sure this is a list of tuples


i = 0
while True:
    well_id = well_ids[i % len(well_ids)]
    channel_xy = channel_positions[i % len(channel_positions)]
    
    well_xy = get_well_xy(plate, well_id, origin_xy)
    print(f"Moving to well {well_id} at {well_xy}")
    s.moveToPos(*well_xy)
    time.sleep(5)
    
    print(f"Moving to channel at {channel_xy}")
    s.moveToPos(*channel_xy)
    time.sleep(5)
    
    i += 1
    # Optional: add a break condition if needed
    if i >= 2: break



Moving to well A1 at (-133645, -8905)
Moving to channel at (-77418, -34524)
Moving to well B1 at (-133645, 95)
Moving to channel at (-77418, -34264)


In [13]:
s.moveToPos(-500,-100)

In [3]:
# Then move printer to first channe;
b = s.getStageXY()

In [23]:
b

[-2060929, -163294]

In [5]:
print((b[0]-a[0], b[1]-a[1]))

(1, 10340)


In [6]:
s.calibPrinterOffset(b[0]-a[0], b[1]-a[1])

In [7]:
s.printerOffset

(1, 6797)

In [8]:
s.firstChannelCamPos

(24354256, -2099863)

In [14]:
s.calibFirstChannelCamPos()

In [12]:
s.firstWellCamPos = s.getStageXY()

In [11]:
s.firstChannelCamPos

(902486, 455375)

In [7]:
# Now move first channel under camera
s.calibFirstChannelCamPos()

In [16]:
s.calibFirstWellCamPos(s.getStageXY())

In [20]:
s.moveWellToPrinter("B1")

In [15]:
s.moveChannelToCam(1)

In [8]:
# Move first well under camera
s.calibFirstWellCamPos(s.getStageXY())

In [15]:
s.moveChannelToCam(25)

In [12]:
s.moveChannelToPrinter(1)

In [13]:
print(ser.read_all())

b''


In [14]:
s.moveWellToPrinter("A1")

In [20]:
s.moveXInUM(-500)

In [4]:
s.close()